In [ ]:
import numpy as np
import matplotlib.pyplot as plt

NPZ_PATH = "/path/to/project/model_dev_main/index_files/min400PM_wind_drift_SAR_dataset_wBackwardPastDrift/sar_hist_train.npz"

def percentile_from_cdf(bin_centers, cdf_row, q):
    target = q / 100.0
    i = np.searchsorted(cdf_row, target, side="left")
    i = int(np.clip(i, 0, len(bin_centers) - 1))
    return float(bin_centers[i])

d = np.load(NPZ_PATH, allow_pickle=True)

names   = [str(x) for x in d["channel_names"]]
edges   = d["bin_edges_db"]
centers = d["bin_centers_db"]
counts  = d["counts"]
pdf     = d["pdf"]
cdf     = d["cdf"]

# New (from revised script)
pdf_mean = d["pdf_mean_across_batches"]  # (C, nbins)
pdf_std  = d["pdf_std_across_batches"]   # (C, nbins)
n_batches_used = int(d["n_batches_used_for_pdf_stats"]) if "n_batches_used_for_pdf_stats" in d else None

meta = d["meta"]

print("Loaded:", NPZ_PATH)
print("Channels:", names)
print("Histogram bins:", len(centers))
if n_batches_used is not None:
    print("Batches used for PDF band stats:", n_batches_used)

print("Meta (summary):")
for k in ["index_path", "dataset_len", "sar_to_db", "db_min", "db_max", "db_step", "n_valid_pixels_per_channel"]:
    if k in meta:
        print(f"  {k}: {meta[k]}")

# Percentiles from GLOBAL CDF
p1s, p99s = [], []
for ci, name in enumerate(names):
    p1  = percentile_from_cdf(centers, cdf[ci], 1)
    p99 = percentile_from_cdf(centers, cdf[ci], 99)
    p1s.append(p1); p99s.append(p99)
    print(f"{name:>12s}  p1={p1:7.2f} dB   p99={p99:7.2f} dB")

#plt.figure(figsize=(9, 5))

for ci, name in enumerate(names):
    p1, p99 = p1s[ci], p99s[ci]

    inside = (centers >= p1) & (centers <= p99)
    left_tail  = centers < p1
    right_tail = centers > p99

    # --- main PDF: inside (colored) ---
    plt.plot(
        centers[inside], pdf[ci][inside],
        linewidth=2,
        label=f"{name} (p1={p1:.1f}, p99={p99:.1f})"
    )

    # --- main PDF: outside (gray), plotted as two tails ---
    plt.plot(centers[left_tail],  pdf[ci][left_tail],  linewidth=2, color="0.6")
    plt.plot(centers[right_tail], pdf[ci][right_tail], linewidth=2, color="0.6")

    # --- std band ---
    lo = np.clip(pdf_mean[ci] - pdf_std[ci], 0.0, None)
    hi = pdf_mean[ci] + pdf_std[ci]

    # inside band (colored)
    plt.fill_between(centers, lo, hi, where=inside, alpha=0.25)

    # outside band (gray) — this is fine as-is (no line-connection issue)
    plt.fill_between(centers, lo, hi, where=left_tail,  color="0.8", alpha=0.35)
    plt.fill_between(centers, lo, hi, where=right_tail, color="0.8", alpha=0.35)

plt.xlabel("Backscatter (dB)")
plt.ylabel("PDF (density)")
plt.title("SAR channel distributions\n(gray = outside 1st–99th percentile range)")
plt.legend()
plt.xlim(-40, 0)
plt.tight_layout()
plt.show()


